# 05 · Zero-shot TimesFM 2.5

`google/timesfm-2.5-200m` on the same 176-origin manifest.

**Key results**
- MAE 6.24 MU vs XGBoost 4.94 — but TimesFM wins day 1 (3.39 vs 4.01) and does so in 58% of origins.
- A horizon-split hybrid (TimesFM day 1, XGBoost days 2–7) gains only 1.8%.
- 56-day context beats 84/112/168.

> Paths are relative to `notebooks/`. Outputs are from the original research run; see [`data/README.md`](../data/README.md) for data that must be downloaded.

➡ Next: [06 · Foundation models vs XGBoost across years](06_foundation_models_walk_forward.ipynb)

Load TimesFM

In [1]:
import numpy as np
import pandas as pd
import timesfm
from pathlib import Path

DATA = Path("../data")

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=56,
        max_horizon=7,
        normalize_inputs=True,
        per_core_batch_size=8,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

print("TimesFM 2.5 loaded")

TimesFM 2.5 loaded


One forecast sanity check

In [2]:
manifest = pd.read_parquet(
    DATA / "processed/gridpulse_eval_176_v1.parquet"
)

energy = pd.read_csv(
    DATA / "processed/delhi_energy_clean.csv",
    parse_dates=["Date"],
    index_col="Date",
).sort_index()["energy_met_MU"]

origin = manifest["origin"].min()

context = energy.loc[
    origin - pd.Timedelta(days=55):origin
].to_numpy(dtype=np.float32)

assert context.shape == (56,)
assert np.isfinite(context).all()

point_forecast, quantiles = model.forecast(
    horizon=7,
    inputs=[context],
)

print("Forecast shape:", point_forecast.shape)
print("Forecast:", point_forecast[0])

actual = (
    manifest[manifest["origin"] == origin]
    .sort_values("horizon")["target"]
    .to_numpy()
)

print("Actual:", actual)
print("MAE:", np.abs(actual - point_forecast[0]).mean())

Forecast shape: (1, 7)
Forecast: [68.48085 66.89369 70.37613 71.39283 73.03428 73.31874 72.78226]
Actual: [58.  59.9 72.4 73.6 73.1 75.3 79.5]
MAE: 4.352900913783481


Batch evaluation

In [3]:
import time
import numpy as np
import pandas as pd

HORIZON = 7
CONTEXT = 56
BATCH_SIZE = 8

# Use the locked evaluation manifest.
manifest = pd.read_parquet(
    DATA / "processed/gridpulse_eval_176_v1.parquet"
).sort_values(["origin", "horizon"])

origins = manifest["origin"].drop_duplicates().tolist()

assert len(origins) == 176
assert len(manifest) == 1232

# Prepare identical historical context for each origin.
contexts = []

for origin in origins:
    history = energy.loc[
        origin - pd.Timedelta(days=CONTEXT-1):origin
    ].to_numpy(dtype=np.float32)

    assert len(history) == CONTEXT
    assert np.isfinite(history).all()

    contexts.append(history)

actuals = (
    manifest["target"]
    .to_numpy()
    .reshape(len(origins), HORIZON)
)

# Run TimesFM inference.
all_predictions = []

start_time = time.perf_counter()

for start in range(0, len(contexts), BATCH_SIZE):
    batch = contexts[start:start+BATCH_SIZE]

    point_forecast, _ = model.forecast(
        horizon=HORIZON,
        inputs=batch,
    )

    all_predictions.append(
        np.asarray(point_forecast)
    )

predictions = np.concatenate(all_predictions, axis=0)

elapsed = time.perf_counter() - start_time

assert predictions.shape == actuals.shape
assert np.isfinite(predictions).all()

errors = actuals - predictions

mae = np.abs(errors).mean()
rmse = np.sqrt(np.mean(errors**2))
wape = 100 * np.abs(errors).sum() / np.abs(actuals).sum()
bias = errors.mean()

print(f"TimesFM MAE:  {mae:.3f} MU")
print(f"TimesFM RMSE: {rmse:.3f} MU")
print(f"TimesFM WAPE: {wape:.2f}%")
print(f"TimesFM Bias: {bias:.3f} MU")
print(f"Inference time: {elapsed:.2f} seconds")
print(f"Seconds per origin: {elapsed/len(origins):.3f}")

# Save one forecast per origin and horizon.
timesfm_results = manifest.copy()
timesfm_results["timesfm_pred"] = predictions.reshape(-1)

output_path = DATA / "processed/timesfm_validation_176_v1.parquet"

timesfm_results.to_parquet(output_path, index=False)

print("Saved:", output_path)

TimesFM MAE:  6.237 MU
TimesFM RMSE: 9.281 MU
TimesFM WAPE: 7.87%
TimesFM Bias: -2.212 MU
Inference time: 3.38 seconds
Seconds per origin: 0.019
Saved: ..\data\processed\timesfm_validation_176_v1.parquet


Analyze TimesFM by horizon

In [4]:
timesfm_results = pd.read_parquet(
    DATA / "processed/timesfm_validation_176_v1.parquet"
)

timesfm_results["abs_error"] = (
    timesfm_results["target"]
    - timesfm_results["timesfm_pred"]
).abs()

by_horizon = (
    timesfm_results.groupby("horizon")["abs_error"]
    .mean()
)

print("TimesFM validation MAE by horizon:")
print(by_horizon.round(3).to_string())

print("\nTimesFM bias by horizon:")
print(
    timesfm_results.assign(
        error=lambda d: d["target"] - d["timesfm_pred"]
    ).groupby("horizon")["error"]
    .mean()
    .round(3)
    .to_string()
)

TimesFM validation MAE by horizon:
horizon
1    3.385
2    4.896
3    5.825
4    6.436
5    7.120
6    7.611
7    8.387

TimesFM bias by horizon:
horizon
1   -0.794
2   -1.629
3   -2.109
4   -2.461
5   -2.537
6   -2.735
7   -3.218


Create a properly paired horizon comparison

In [5]:
xgb = pd.read_parquet(
    DATA / "processed/xgb_validation_v01.parquet"
)

timesfm = pd.read_parquet(
    DATA / "processed/timesfm_validation_176_v1.parquet"
)

keys = ["origin", "target_date", "horizon"]

paired = timesfm.merge(
    xgb[keys + ["xgb_pred"]],
    on=keys,
    how="inner",
    validate="one_to_one",
)

assert len(paired) == 1232

paired["timesfm_abs_error"] = (
    paired["target"] - paired["timesfm_pred"]
).abs()

paired["xgb_abs_error"] = (
    paired["target"] - paired["xgb_pred"]
).abs()

horizon_comparison = paired.groupby("horizon")[
    ["timesfm_abs_error", "xgb_abs_error"]
].mean()

horizon_comparison["timesfm_advantage_pct"] = (
    100
    * (
        horizon_comparison["xgb_abs_error"]
        - horizon_comparison["timesfm_abs_error"]
    )
    / horizon_comparison["xgb_abs_error"]
)

display(horizon_comparison.round(3))

,timesfm_abs_error,xgb_abs_error,timesfm_advantage_pct
horizon,,,
1,3.385,4.011,15.593
2,4.896,4.446,-10.112
3,5.825,4.696,-24.044
4,6.436,4.972,-29.448
5,7.120,5.193,-37.116
6,7.611,5.447,-39.726
7,8.387,5.811,-44.330


Horizon-specific hybrid - 
TimesFM Forecast Day 1
XGBoost Forecast Days 2–7

In [6]:
hybrid = paired.copy()

hybrid["hybrid_pred"] = np.where(
    hybrid["horizon"] == 1,
    hybrid["timesfm_pred"],
    hybrid["xgb_pred"]
)

def mae(actual, predicted):
    return np.abs(actual - predicted).mean()

summary = pd.DataFrame({
    "Model": ["XGBoost", "TimesFM", "Hybrid"],
    "MAE": [
        mae(hybrid["target"], hybrid["xgb_pred"]),
        mae(hybrid["target"], hybrid["timesfm_pred"]),
        mae(hybrid["target"], hybrid["hybrid_pred"]),
    ],
})

display(summary.round(3))

print(
    "Hybrid improvement vs XGBoost:",
    f"{100 * (summary.loc[0, 'MAE'] - summary.loc[2, 'MAE']) / summary.loc[0, 'MAE']:.2f}%"
)

,Model,MAE
0,XGBoost,4.939
1,TimesFM,6.237
2,Hybrid,4.850


Hybrid improvement vs XGBoost: 1.81%


Is TimesFM's Day-1 advantage consistent?

In [7]:
day1 = paired[paired["horizon"] == 1].copy()

day1["month"] = day1["target_date"].dt.month

day1["timesfm_error"] = (
    day1["target"] - day1["timesfm_pred"]
).abs()

day1["xgb_error"] = (
    day1["target"] - day1["xgb_pred"]
).abs()

day1["timesfm_wins"] = (
    day1["timesfm_error"] < day1["xgb_error"]
)

print("TimesFM Day-1 win rate:",
      f"{day1['timesfm_wins'].mean():.2%}")

print("\nDay-1 performance by month:")

monthly = day1.groupby("month").agg(
    origins=("origin", "count"),
    timesfm_mae=("timesfm_error", "mean"),
    xgb_mae=("xgb_error", "mean"),
    timesfm_win_rate=("timesfm_wins", "mean"),
)

monthly["advantage_pct"] = (
    100
    * (monthly["xgb_mae"] - monthly["timesfm_mae"])
    / monthly["xgb_mae"]
)

display(monthly.round(3))

TimesFM Day-1 win rate: 57.95%

Day-1 performance by month:


,origins,timesfm_mae,xgb_mae,timesfm_win_rate,advantage_pct
month,,,,,
1,24,3.518,3.876,0.542,9.233
2,28,1.568,1.224,0.464,-28.120
3,31,3.171,3.344,0.613,5.182
4,15,2.381,4.012,0.800,40.668
9,24,6.313,7.451,0.458,15.269
10,31,4.472,5.688,0.677,21.377
11,23,1.885,2.593,0.565,27.294


TimesFM context length - 56, 84, 112 and 168 days

In [8]:
CONTEXT_LENGTHS = [56, 84, 112, 168]

origins = (
    manifest["origin"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

eligibility = []

for origin in origins:
    row = {"origin": origin}

    for length in CONTEXT_LENGTHS:
        history = energy.loc[
            origin - pd.Timedelta(days=length-1):origin
        ]

        row[f"context_{length}"] = (
            len(history) == length
            and history.notna().all()
        )

    eligibility.append(row)

eligibility = pd.DataFrame(eligibility)

print("Eligible origins:")
for length in CONTEXT_LENGTHS:
    print(
        f"{length} days:",
        eligibility[f"context_{length}"].sum()
    )

Eligible origins:
56 days: 176
84 days: 176
112 days: 120
168 days: 14


In [11]:
import timesfm

print(type(timesfm))
print(timesfm.ForecastConfig)

<class 'module'>
<class 'timesfm.configs.ForecastConfig'>


In [12]:
import time
import numpy as np
import pandas as pd

common_origins = (
    eligibility.loc[eligibility["context_112"], "origin"]
    .sort_values()
    .tolist()
)

assert len(common_origins) == 120

HORIZON = 7
BATCH_SIZE = 8

# Load actual targets in a consistent order.
eval_rows = (
    manifest[manifest["origin"].isin(common_origins)]
    .sort_values(["origin", "horizon"])
)

actuals = eval_rows["target"].to_numpy().reshape(120, 7)

results = []
context_predictions = {}

for length in [56, 84, 112]:

    model.compile(
        timesfm.ForecastConfig(
            max_context=length,
            max_horizon=HORIZON,
            normalize_inputs=True,
            per_core_batch_size=8,
            use_continuous_quantile_head=True,
            force_flip_invariance=True,
            infer_is_positive=True,
            fix_quantile_crossing=True,
        )
    )

    contexts = [
        energy.loc[
            origin - pd.Timedelta(days=length - 1):origin
        ].to_numpy(dtype=np.float32)
        for origin in common_origins
    ]

    all_preds = []
    start_time = time.perf_counter()

    for start in range(0, len(contexts), BATCH_SIZE):
        batch = contexts[start:start + BATCH_SIZE]

        forecasts, _ = model.forecast(
            horizon=HORIZON,
            inputs=batch,
        )

        all_preds.append(np.asarray(forecasts))

    elapsed = time.perf_counter() - start_time

    predictions = np.concatenate(all_preds, axis=0)

    assert predictions.shape == actuals.shape

    context_predictions[length] = predictions

    errors = actuals - predictions

    results.append({
        "context": length,
        "MAE": np.abs(errors).mean(),
        "RMSE": np.sqrt(np.mean(errors**2)),
        "Bias": errors.mean(),
        "Day_1_MAE": np.abs(errors[:, 0]).mean(),
        "Day_7_MAE": np.abs(errors[:, 6]).mean(),
        "Inference_seconds": elapsed,
    })

context_results = pd.DataFrame(results).set_index("context")

display(context_results.round(3))

,MAE,RMSE,Bias,Day_1_MAE,Day_7_MAE,Inference_seconds
context,,,,,,
56,5.393,7.842,-0.932,2.845,7.040,2.361
84,5.915,8.171,-0.417,3.082,7.995,2.485
112,6.146,8.624,-0.195,3.072,8.306,2.731


Compare TimesFM and XGBoost on the same 120 origins

In [13]:
# Existing XGBoost predictions
xgb = pd.read_parquet(
    DATA / "processed/xgb_validation_v01.parquet"
)

# Exactly the 120 origins from the context experiment
common = xgb[xgb["origin"].isin(common_origins)].copy()

assert common["origin"].nunique() == 120
assert len(common) == 120 * 7

xgb_mae = np.abs(
    common["target"] - common["xgb_pred"]
).mean()

timesfm_mae = context_results.loc[56, "MAE"]

print(f"XGBoost MAE (120 origins): {xgb_mae:.3f} MU")
print(f"TimesFM MAE (120 origins): {timesfm_mae:.3f} MU")

print(
    "TimesFM improvement over XGBoost:",
    f"{100 * (xgb_mae - timesfm_mae) / xgb_mae:+.2f}%"
)

XGBoost MAE (120 origins): 4.452 MU
TimesFM MAE (120 origins): 5.393 MU
TimesFM improvement over XGBoost: -21.14%
